# SPIS-ML LSTM Training en Google Colab

Entrena LSTM con dataset sísmico 605MB. Exporta modelo .h5 para dashboard.


In [ ]:
!pip install -q pandas numpy scikit-learn tensorflow matplotlib plotly

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive montado')

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Nadam
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

## 1. Cargar Datos


In [ ]:
# CAMBIAR ESTA RUTA
DATA_PATH = '/content/drive/My Drive/SPIS-ML/sismos.csv'

print(f'Cargando: {DATA_PATH}')

if DATA_PATH.endswith('.parquet'):
    df = pd.read_parquet(DATA_PATH)
else:
    df = pd.read_csv(DATA_PATH)

print(f'Shape: {df.shape}')
print(f'Columnas: {list(df.columns)}')
print(df.head())

In [ ]:
# Detectar columna magnitud
mag_col = None
for col in df.columns:
    if 'mag' in col.lower():
        mag_col = col
        break

if mag_col is None:
    print('COLUMNAS DISPONIBLES:', list(df.columns))
    mag_col = 'mag'
else:
    print(f'Columna magnitud: {mag_col}')

print(df[mag_col].describe())

## 2. Preprocesar


In [ ]:
series = df[mag_col].dropna().values.reshape(-1, 1).astype('float32')
print(f'Muestras: {len(series)}')

scaler = MinMaxScaler(feature_range=(0, 1))
dataset_scaled = scaler.fit_transform(series)

print(f'Min: {dataset_scaled.min():.4f}')
print(f'Max: {dataset_scaled.max():.4f}')

plt.figure(figsize=(12, 3))
plt.subplot(1, 2, 1)
plt.hist(series, bins=50, alpha=0.7)
plt.title('Distribucion magnitudes')
plt.subplot(1, 2, 2)
plt.plot(series[:1000])
plt.title('Primeros 1000')
plt.show()

## 3. Secuencias Temporales


In [ ]:
def create_dataset(dataset, look_back=100):
    dataX, dataY = [], []
    for i in range(len(dataset) - look_back - 1):
        a = dataset[i:(i + look_back), 0]
        dataX.append(a)
        dataY.append(dataset[i + look_back, 0])
    return np.array(dataX), np.array(dataY)

LOOK_BACK = 100
TRAIN_SPLIT = 0.8

train_size = int(len(dataset_scaled) * TRAIN_SPLIT)
train_data = dataset_scaled[:train_size, :]
test_data = dataset_scaled[train_size:, :]

trainX, trainY = create_dataset(train_data, LOOK_BACK)
testX, testY = create_dataset(test_data, LOOK_BACK)

trainX = np.reshape(trainX, (trainX.shape[0], LOOK_BACK, 1))
testX = np.reshape(testX, (testX.shape[0], LOOK_BACK, 1))

print(f'trainX: {trainX.shape}')
print(f'testX: {testX.shape}')

## 4. Entrenar LSTM


In [ ]:
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(LOOK_BACK, 1), activation='relu'),
    Dropout(0.2),
    LSTM(32, activation='relu'),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])

model.compile(
    optimizer=Nadam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

print(model.summary())

In [ ]:
EPOCHS = 50
BATCH_SIZE = 32

print(f'Entrenando {EPOCHS} epochs...')

history = model.fit(
    trainX, trainY,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    verbose=1,
    shuffle=True
)

print('Entrenamiento completado')

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.title('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history['mae'], label='Train')
plt.plot(history.history['val_mae'], label='Val')
plt.title('MAE')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Evaluar


In [ ]:
print('Prediciendo...')
train_predict = model.predict(trainX, verbose=0)
test_predict = model.predict(testX, verbose=0)

train_predict_real = scaler.inverse_transform(train_predict)
test_predict_real = scaler.inverse_transform(test_predict)
trainY_real = scaler.inverse_transform(trainY.reshape(-1, 1))
testY_real = scaler.inverse_transform(testY.reshape(-1, 1))

train_rmse = np.sqrt(mean_squared_error(trainY_real, train_predict_real))
test_rmse = np.sqrt(mean_squared_error(testY_real, test_predict_real))
train_mae = mean_absolute_error(trainY_real, train_predict_real)
test_mae = mean_absolute_error(testY_real, test_predict_real)
train_r2 = r2_score(trainY_real, train_predict_real)
test_r2 = r2_score(testY_real, test_predict_real)

print(f'Train RMSE: {train_rmse:.4f}')
print(f'Train MAE:  {train_mae:.4f}')
print(f'Train R2:   {train_r2:.4f}')
print()
print(f'Test RMSE: {test_rmse:.4f}')
print(f'Test MAE:  {test_mae:.4f}')
print(f'Test R2:   {test_r2:.4f}')

In [ ]:
n_show = min(300, len(testY_real))

plt.figure(figsize=(14, 5))
plt.plot(testY_real[-n_show:], label='Real', linewidth=2)
plt.plot(test_predict_real[-n_show:], label='Prediccion', linewidth=2, linestyle='--')
plt.title(f'Ultimos {n_show} eventos')
plt.xlabel('Indice')
plt.ylabel('Magnitud')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(8, 8))
plt.scatter(testY_real, test_predict_real, alpha=0.5, s=20)
min_val = min(testY_real.min(), test_predict_real.min())
max_val = max(testY_real.max(), test_predict_real.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)
plt.xlabel('Real')
plt.ylabel('Prediccion')
plt.grid(True, alpha=0.3)
plt.show()

## 6. Exportar H5


In [ ]:
import os

OUTPUT_DIR = '/content/drive/My Drive/SPIS-ML'
MODEL_PATH = f'{OUTPUT_DIR}/lstm_trained.h5'

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Guardando en: {MODEL_PATH}')
model.save(MODEL_PATH)
print('Guardado')

model_size = os.path.getsize(MODEL_PATH) / (1024**2)
print(f'Tamanio: {model_size:.2f} MB')

## 7. Test: Cargar Modelo


In [ ]:
print(f'Cargando desde: {MODEL_PATH}')
loaded_model = tf.keras.models.load_model(MODEL_PATH)
print('Cargado OK')

test_sample = testX[:5]
predictions = loaded_model.predict(test_sample, verbose=0)
predictions_real = scaler.inverse_transform(predictions)

print('\nTest prediccion:')
for i, (pred, real) in enumerate(zip(predictions_real, testY_real[:5])):
    error = abs(pred[0] - real[0])
    print(f'  {i}: pred={pred[0]:.3f}, real={real[0]:.3f}, error={error:.3f}')

## 8. Integrar en Dashboard

1. Descargar lstm_trained.h5 desde Drive
2. Copiar a: dashboard/models/lstm_trained.h5
3. Editar model_lstm_prediction_module.py:

```python
from tensorflow.keras.models import load_model

MODEL_PATH = 'models/lstm_trained.h5'
model = load_model(MODEL_PATH)  # Load una sola vez

def run_lstm_analysis(df, magnitude_col='mag'):
    series = df[magnitude_col].dropna().values.reshape(-1, 1).astype('float32')
    scaler = MinMaxScaler(feature_range=(0, 1))
    dataset_scaled = scaler.fit_transform(series)
    
    look_back = 100
    train_size = int(len(dataset_scaled) * 0.7)
    test = dataset_scaled[train_size:len(dataset_scaled),:]
    
    trainX, trainY = create_dataset(dataset_scaled[:train_size], look_back)
    testX, testY = create_dataset(test, look_back)
    testX = np.reshape(testX, (testX.shape[0], look_back, 1))
    
    # Solo predecir
    test_predict = model.predict(testX)
    test_predict = scaler.inverse_transform(test_predict)
    y_test_real = scaler.inverse_transform(testY.reshape(-1, 1))
    
    rmse = np.sqrt(mean_squared_error(y_test_real, test_predict))
    r2 = r2_score(y_test_real, test_predict)
    
    # ... resto igual ...
```

4. Ejecutar: `python app.py`
